In [0]:
from pyspark.sql import functions as f

CATALOG = "workspace"

patients = spark.read.table(f"{CATALOG}.hc_bronze.patients")

#1.convert text to real dates
patients = patients.withColumn("BIRTHDATE", f.to_date("BIRTHDATE"))
patients = patients.withColumn("DEATHDATE", f.to_date("DEATHDATE"))

#2 Age: up to death date if deceased , otherwise up to today
end_date = f.coalesce("DEATHDATE", f.current_date())
patients = patients.withColumn("AGE",f.floor(f.datediff(end_date, "BIRTHDATE") / 365.25))

#3. Flag and age group
patients = patients.withColumn("is_deceased",f.col("DEATHDATE").isNotNull())
patients = patients.withColumn("AGE_GROUP",
                                f.when(f.col("AGE") < 10, "0-9")
                                .when(f.col("AGE") < 20, "10-19")
                                .when(f.col("AGE") < 30, "20-29")
                                .when(f.col("AGE") < 40, "30-39")
                                .when(f.col("AGE") < 50, "40-49")
                                .when(f.col("AGE") < 60, "50-59")
                                .when(f.col("AGE") < 70, "60-69")
                                .otherwise("70+")
                               )

#4 data quality check
bad_dates = patients.filter(f.col("DEATHDATE") < f.col("BIRTHDATE")).count()
assert bad_dates == 0, f"Found {bad_dates} patients with death date before birth date"
total = patients.count()
assert total > 0, f"Found {total} patients"
distinct_ids = patients.select("ID").distinct().count()
assert distinct_ids == total, f"Found {distinct_ids} distinct ids, expected {total} unique ids"

#5. write to silver
patients.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.hc_silver.patients")


In [0]:
%sql
select age_group,count(*) from workspace.hc_silver.patients group by age_group order by age_group;